In [1]:
# =========================================================================
# 大型拠点ダッシュボード 集計バッチ処理 (Google Colab Python版 / 5階層ETL対応)
# =========================================================================
import io
import csv
import datetime
import calendar
import re
from dateutil.relativedelta import relativedelta
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

# 1. 認証処理
print("🔑 Google Driveに接続して認証を行っています...")
auth.authenticate_user()
drive_service = build('drive', 'v3')
print("✅ 認証成功")

# =========================================================================
# 2. 設定エリア
# =========================================================================
TARGET_START = '2026-03-01'
TARGET_END   = '2026-03-30'

# ★ 5階層の出力フォルダID
OUTPUT_FOLDERS = {
    'daily':           '1VOUIgT45cVMiP4JYp8o7yiQkTQEwKl4K', # 01.日次データ
    'weekly':          '1k3cJB1Rn4DwXIBg-DKhO80HiXxJKP4oG', # 02.週次データ
    'monthly':         '17JzLXnliEYHSqpAbjVbwKThUYGzvVJu8', # 03.月次データ
    'weekly_running':  '1LgUwCFQe1nl5_ktnpA7o6Q52B0Wh5rfk', # 04.週累計
    'monthly_running': '1imvPQkJHRApG2Qk2VTVDy2teemLRahbc'  # 05.月累計
}

INPUT_FOLDERS = {
    'workVolume':  {'prefix': '集約拠点_作業個数_時間帯別_',       'id': '1llD95sgV5c7Cg_OiWOjTQz3NBb1gFyWO'},
    'manpower':    {'prefix': '集約拠点_人員数_投入時間_時間帯別_', 'id': '1HIrPENYqUB7U1jJq9WXstsVuKYO51hqB'},
    'outsourcing': {'prefix': '集約拠点_外部リソース_',             'id': '1g38uOUeEYEau4GMcXgG18hQwmIfOqeZP'},
    'cost':        {'prefix': 'YTCBIZ人的コスト_歴月収支有_抜粋版_', 'id': '17Z5k-DPJIR9nYMjPtRBq1PmBJonP9lLO'},
    'timee':       {'prefix': '', 'suffix': '_タイミー実績.csv',   'id': '1yl8nnhWL_U7kUYZEU4ewXN0Nr9I1sp8I'}
}

PERIOD_HEADERS = ['基準日', '集計対象期間開始', '集計対象期間終了', '実績集計期間開始', '実績集計期間終了', '集計対象日数', 'データ存在日数', '拠点コード', '拠点名称', '時間帯', '作業個数_合計', '作業個数_日平均', '総投入時間(分)_合計', '総投入時間(分)_日平均', '自社投入時間(分)_合計', '自社投入時間(分)_日平均', 'タイミー投入時間(分)_合計', 'タイミー投入時間(分)_日平均', '外部投入時間(分)_合計', '外部投入時間(分)_日平均', '総人員数_合計', '総人員数_日平均', '自社人員数_合計', '自社人員数_日平均', 'タイミー人員数_合計', 'タイミー人員数_日平均', '外部人員数_合計', '外部人員数_日平均', '総コスト_合計', '総コスト_日平均', '自社コスト_合計', '自社コスト_日平均', 'タイミーコスト_合計', 'タイミーコスト_日平均', '外部コスト_合計', '外部コスト_日平均', '個当たりコスト_合計ベース']

# =========================================================================
# 3. Drive操作ヘルパー
# =========================================================================
def get_files_in_folder(folder_id):
    query = f"'{folder_id}' in parents and trashed=false"
    files = []
    page_token = None
    while True:
        response = drive_service.files().list(q=query, spaces='drive', fields='nextPageToken, files(id, name, modifiedTime)', includeItemsFromAllDrives=True, supportsAllDrives=True, pageToken=page_token).execute()
        files.extend(response.get('files', []))
        page_token = response.get('nextPageToken', None)
        if page_token is None: break
    return files

def download_and_parse_csv(file_id, file_name=""):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done: status, done = downloader.next_chunk()
    fh.seek(0)
    byte_content = fh.read()
    text = ""
    try:
        text = byte_content.decode('utf-8-sig')
        if not any(k in text for k in ['拠点', '時間', 'コード', '社員', '月']): raise ValueError("UTF-8 but no keywords")
    except:
        try: text = byte_content.decode('cp932')
        except: return []
    if not text: return []
    reader = csv.DictReader(io.StringIO(text))
    return [{k.lstrip('\ufeff').strip() if k else k: v.strip() if isinstance(v, str) else v for k, v in row.items()} for row in reader]

def upload_csv(folder_id, filename, csv_rows, headers):
    query = f"'{folder_id}' in parents and name='{filename}' and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id)", includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get('files', [])
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(headers)
    writer.writerows(csv_rows)
    media = MediaIoBaseUpload(io.BytesIO(output.getvalue().encode('utf-8-sig')), mimetype='text/csv')
    if results: drive_service.files().update(fileId=results[0]['id'], media_body=media, supportsAllDrives=True).execute()
    else: drive_service.files().create(body={'name': filename, 'parents': [folder_id]}, media_body=media, supportsAllDrives=True).execute()

def get_num(row, key):
    try: return float(row.get(key, 0) or 0)
    except: return 0.0
def get_int(row, key):
    try: return int(row.get(key, 0) or 0)
    except: return 0

# =========================================================================
# 4. 集計ロジック (日次)
# =========================================================================
def create_daily_csv(date_str, source_caches):
    ds_no_hyphen = date_str.replace('-', '')
    raw = {}
    cost_file_name = "不明"

    for key, c_data in source_caches.items():
        if key == 'cost':
            target_date = datetime.datetime.strptime(date_str, '%Y-%m-%d')
            found_cost = False
            for i in range(12):
                chk_date = target_date - relativedelta(months=i)
                ym = chk_date.strftime('%Y%m')
                target_name = f"{INPUT_FOLDERS['cost']['prefix']}{ym}.csv"
                match = [f for f in c_data if f['name'] == target_name]
                if match:
                    raw['cost'] = download_and_parse_csv(match[0]['id'])
                    cost_file_name = match[0]['name']
                    found_cost = True
                    break
            if not found_cost:
                valid_files = [f for f in c_data if f['name'].startswith(INPUT_FOLDERS['cost']['prefix'])]
                if valid_files:
                    valid_files.sort(key=lambda x: x.get('modifiedTime', ''), reverse=True)
                    raw['cost'] = download_and_parse_csv(valid_files[0]['id'])
                    cost_file_name = valid_files[0]['name']
                else: raw['cost'] = []
            continue

        prefix = INPUT_FOLDERS[key]['prefix']
        if key == 'timee': prefix = ds_no_hyphen
        suffix = INPUT_FOLDERS[key].get('suffix', '.csv')
        possible_names = [f"{prefix}{date_str}{suffix}", f"{prefix}{ds_no_hyphen}{suffix}"]
        if key == 'timee': possible_names = [f"{ds_no_hyphen}{suffix}", f"{date_str}{suffix}"]
        match = [f for f in c_data if f['name'] in possible_names]
        if not match: match = [f for f in c_data if (date_str in f['name'] or ds_no_hyphen in f['name']) and f['name'].endswith('.csv')]
        if match: raw[key] = download_and_parse_csv(match[0]['id'], match[0]['name'])
        else: raw[key] = []

    name_map, cost_map, agg_map, vol_map = {}, {}, {}, {}
    for k in ['workVolume', 'manpower', 'outsourcing', 'timee']:
        for r in raw.get(k, []):
            code = r.get('集約拠点コード') or r.get('拠点コード')
            name = r.get('拠点名称')
            if code and name and not str(name).startswith('('): name_map[code] = name

    for r in raw.get('cost', []):
        code, staff, wage = r.get('集約拠点コード') or r.get('拠点コード'), r.get('社員区分名称'), get_num(r, '単純時給')
        if code and staff and wage > 0:
            if code not in cost_map: cost_map[code] = {}
            cost_map[code][staff] = wage

    def get_entry(c, t):
        k = f"{c}-{t}"
        if k not in agg_map: agg_map[k] = {'internal': {'h':0, 'c':0, 'v':0}, 'timee': {'h':0, 'c':0, 'v':0}, 'outsourcing': {'h':0, 'c':0, 'v':0}}
        return agg_map[k]

    staff_map = {'YSS':'ＹＳＳ作業', 'パート':'パート作業', 'アルバイト':'アルバイト作業', 'フル':'フル作業'}
    valid_codes = set(name_map.keys())

    for r in raw.get('manpower', []):
        code, time_slot, staff = r.get('集約拠点コード') or r.get('拠点コード'), r.get('時間帯'), r.get('社員区分名称')
        if not code or not time_slot or not staff: continue
        e, m = get_entry(code, time_slot), get_num(r, '投入時間')
        e['internal']['h'] += m
        e['internal']['c'] += get_int(r, '人員数') or get_int(r, '投入人数')
        m_type = staff_map.get(staff)
        if m_type and code in cost_map and m_type in cost_map[code]: e['internal']['v'] += (m/60.0) * cost_map[code][m_type]

    for k_type, raw_key in [('timee', 'timee'), ('outsourcing', 'outsourcing')]:
        for r in raw.get(raw_key, []):
            code, time_slot = r.get('集約拠点コード') or r.get('拠点コード'), r.get('時間帯')
            if not code or not time_slot: continue
            if code not in valid_codes and str(int(code) if str(code).isdigit() else code) in valid_codes: code = str(int(code))
            e = get_entry(code, time_slot)
            e[k_type]['h'] += get_num(r, '投入時間') * 60
            e[k_type]['c'] += get_int(r, '投入人数') or get_int(r, '人員数')
            e[k_type]['v'] += get_num(r, '人的コスト') if raw_key == 'outsourcing' else get_num(r, '合計支払金額')

    for r in raw.get('workVolume', []):
        code, time_slot = r.get('集約拠点コード') or r.get('拠点コード'), r.get('時間帯')
        vol_map[f"{code}-{time_slot}"] = get_int(r, '作業個数')

    headers = ['対象日', '拠点コード', '拠点名称', '時間帯', '作業個数', '総投入時間(分)', '自社投入時間(分)', 'タイミー投入時間(分)', '外部投入時間(分)', '総人員数', '自社人員数', 'タイミー人員数', '外部人員数', '総コスト', '自社コスト', 'タイミーコスト', '外部コスト', '個当たりコスト', '参照コストファイル']
    csv_rows = []
    for k in set(list(vol_map.keys()) + list(agg_map.keys())):
        code, time_slot = k.split('-', 1)
        name = name_map.get(code, f"(名称不明:{code})")
        if str(name).startswith('(名称不明') or str(name).startswith('(不明'): continue
        vol = vol_map.get(k, 0)
        e = agg_map.get(k, {'internal':{'h':0,'c':0,'v':0}, 'timee':{'h':0,'c':0,'v':0}, 'outsourcing':{'h':0,'c':0,'v':0}})
        if vol == 0 and e['internal']['h']==0 and e['timee']['h']==0 and e['outsourcing']['h']==0: continue
        th = e['internal']['h'] + e['timee']['h'] + e['outsourcing']['h']
        tc = e['internal']['c'] + e['timee']['c'] + e['outsourcing']['c']
        tv = e['internal']['v'] + e['timee']['v'] + e['outsourcing']['v']
        cpi = tv / vol if vol > 0 else 0
        csv_rows.append([date_str, code, name, time_slot, vol, round(th, 1), round(e['internal']['h'], 1), round(e['timee']['h'], 1), round(e['outsourcing']['h'], 1), tc, e['internal']['c'], e['timee']['c'], e['outsourcing']['c'], round(tv, 0), round(e['internal']['v'], 0), round(e['timee']['v'], 0), round(e['outsourcing']['v'], 0), round(cpi, 1), cost_file_name])
    upload_csv(OUTPUT_FOLDERS['daily'], f"集約拠点_ダッシュボード集計_{date_str}.csv", csv_rows, headers)

# =========================================================================
# 5. 集計ロジック (週次・月次・累計)
# =========================================================================
def build_and_upload_period_csv(date_list, daily_files_cache, ref_date, start_d, end_d, total_days, folder_id, prefix, filename_date):
    period_data = {}
    for d_str in date_list:
        match = [f for f in daily_files_cache if f['name'] == f"集約拠点_ダッシュボード集計_{d_str}.csv"]
        if not match: continue
        for r in download_and_parse_csv(match[0]['id']):
            code, ts, name = r.get('拠点コード'), r.get('時間帯'), r.get('拠点名称')
            if not code or not ts or str(name).startswith('(名称') or str(name).startswith('(不明'): continue
            k = f"{code}-{ts}"
            if k not in period_data: period_data[k] = {'code': code, 'name': name, 'timeSlot': ts, 'dates': set(), 'entries': []}
            period_data[k]['dates'].add(d_str)
            period_data[k]['entries'].append({'vol': get_num(r, '作業個数'), 'th': get_num(r, '総投入時間(分)'), 'ih': get_num(r, '自社投入時間(分)'), 'tmh': get_num(r, 'タイミー投入時間(分)'), 'oh': get_num(r, '外部投入時間(分)'), 'tc': get_num(r, '総人員数'), 'ic': get_num(r, '自社人員数'), 'tmc': get_num(r, 'タイミー人員数'), 'oc': get_num(r, '外部人員数'), 'tv': get_num(r, '総コスト'), 'iv': get_num(r, '自社コスト'), 'tmv': get_num(r, 'タイミーコスト'), 'ov': get_num(r, '外部コスト')})

    csv_rows = []
    for pd in period_data.values():
        ddays = len(pd['dates'])
        if ddays == 0: continue
        actual_start = min(pd['dates'])
        actual_end = max(pd['dates'])
        sum_v = {k: sum(e[k] for e in pd['entries']) for k in ['vol','th','ih','tmh','oh','tc','ic','tmc','oc','tv','iv','tmv','ov']}
        avg_v = {k: v / ddays for k, v in sum_v.items()}
        cpi_tot = sum_v['tv'] / sum_v['vol'] if sum_v['vol'] > 0 else 0
        csv_rows.append([ref_date, start_d, end_d, actual_start, actual_end, total_days, ddays, pd['code'], pd['name'], pd['timeSlot'], sum_v['vol'], round(avg_v['vol'],1), round(sum_v['th'],1), round(avg_v['th'],1), round(sum_v['ih'],1), round(avg_v['ih'],1), round(sum_v['tmh'],1), round(avg_v['tmh'],1), round(sum_v['oh'],1), round(avg_v['oh'],1), sum_v['tc'], round(avg_v['tc'],1), sum_v['ic'], round(avg_v['ic'],1), sum_v['tmc'], round(avg_v['tmc'],1), sum_v['oc'], round(avg_v['oc'],1), round(sum_v['tv'],1), round(avg_v['tv'],1), round(sum_v['iv'],1), round(avg_v['iv'],1), round(sum_v['tmv'],1), round(avg_v['tmv'],1), round(sum_v['ov'],1), round(avg_v['ov'],1), round(cpi_tot,1)])
    upload_csv(folder_id, f"{prefix}{filename_date}.csv", csv_rows, PERIOD_HEADERS)

def build_running_from_period_file(source_folder_id, source_filename, ref_date, end_d, target_folder_id, target_filename):
    source_files = get_files_in_folder(source_folder_id)
    match = [f for f in source_files if f['name'] == source_filename]
    if not match:
        print(f"  ⚠️ 累計元ファイルなし: {source_filename}")
        return

    source_rows = download_and_parse_csv(match[0]['id'])
    csv_rows = []
    for r in source_rows:
        r['基準日'] = ref_date
        if (r.get('実績集計期間終了') or '') > end_d:
            r['実績集計期間終了'] = end_d
        csv_rows.append([r.get(h, '') for h in PERIOD_HEADERS])

    upload_csv(target_folder_id, target_filename, csv_rows, PERIOD_HEADERS)

def main():
    start_dt = datetime.datetime.strptime(TARGET_START, '%Y-%m-%d')
    end_dt = datetime.datetime.strptime(TARGET_END, '%Y-%m-%d')
    actual_start = min(start_dt.replace(day=1), start_dt - datetime.timedelta(days=start_dt.weekday()))

    print("📂 Drive内のファイルリストを取得中...")
    source_caches = {k: get_files_in_folder(v['id']) for k, v in INPUT_FOLDERS.items()}

    date_pattern = re.compile(r'(\d{4}-\d{2}-\d{2})')
    valid_dates = sorted(list(set([match.group(1) for f in source_caches['workVolume'] if (match := date_pattern.search(f['name'])) and actual_start <= datetime.datetime.strptime(match.group(1), '%Y-%m-%d') <= end_dt])))

    if not valid_dates: return print("❌ 対象データなし")
    print(f"🚀 全 {len(valid_dates)} 日分の集計開始...")

    for i, d_str in enumerate(valid_dates):
        print(f"\n--- [{i+1}/{len(valid_dates)}] {d_str} ---")
        # 1. 日次
        create_daily_csv(d_str, source_caches)

        daily_files_cache = get_files_in_folder(OUTPUT_FOLDERS['daily'])
        tgt = datetime.datetime.strptime(d_str, '%Y-%m-%d')

        # 準備
        mon = tgt - datetime.timedelta(days=tgt.weekday())
        sun = mon + datetime.timedelta(days=6)
        fst = tgt.replace(day=1)
        last_day = calendar.monthrange(tgt.year, tgt.month)[1]

        week_full = [(mon + datetime.timedelta(days=x)).strftime('%Y-%m-%d') for x in range(7)]
        month_full = [(fst + datetime.timedelta(days=x)).strftime('%Y-%m-%d') for x in range(last_day)]

        # 2. 週次・月次（元データ作成）
        week_start_str = mon.strftime('%Y-%m-%d')
        month_start_str = fst.strftime('%Y-%m')
        build_and_upload_period_csv(week_full, daily_files_cache, d_str, week_start_str, sun.strftime('%Y-%m-%d'), 7, OUTPUT_FOLDERS['weekly'], '集約拠点_ダッシュボード週次_', week_start_str)
        build_and_upload_period_csv(month_full, daily_files_cache, d_str, fst.strftime('%Y-%m-%d'), fst.replace(day=last_day).strftime('%Y-%m-%d'), last_day, OUTPUT_FOLDERS['monthly'], '集約拠点_ダッシュボード月次_', month_start_str)

        # 3. 累計（週次・月次データを参照して作成）
        build_running_from_period_file(OUTPUT_FOLDERS['weekly'], f"集約拠点_ダッシュボード週次_{week_start_str}.csv", d_str, d_str, OUTPUT_FOLDERS['weekly_running'], f"集約拠点_ダッシュボード週累計_{week_start_str}.csv")
        build_running_from_period_file(OUTPUT_FOLDERS['monthly'], f"集約拠点_ダッシュボード月次_{month_start_str}.csv", d_str, d_str, OUTPUT_FOLDERS['monthly_running'], f"集約拠点_ダッシュボード月累計_{month_start_str}.csv")

        print("  ✅ 5階層完了")

main()

✅ 認証成功
📂 Drive内のファイルリストを取得中...
🚀 全 30 日分の集計開始...

--- [1/30] 2026-02-23 ---
  ✅ 5階層完了

--- [2/30] 2026-02-24 ---
  ✅ 5階層完了

--- [3/30] 2026-02-25 ---
  ✅ 5階層完了

--- [4/30] 2026-02-26 ---
  ✅ 5階層完了

--- [5/30] 2026-02-27 ---
  ✅ 5階層完了

--- [6/30] 2026-02-28 ---
  ✅ 5階層完了

--- [7/30] 2026-03-01 ---
  ✅ 5階層完了

--- [8/30] 2026-03-02 ---
  ✅ 5階層完了

--- [9/30] 2026-03-03 ---
  ✅ 5階層完了

--- [10/30] 2026-03-04 ---
  ✅ 5階層完了

--- [11/30] 2026-03-05 ---
  ✅ 5階層完了

--- [12/30] 2026-03-06 ---
  ✅ 5階層完了

--- [13/30] 2026-03-07 ---
  ✅ 5階層完了

--- [14/30] 2026-03-08 ---
  ✅ 5階層完了

--- [15/30] 2026-03-09 ---
  ✅ 5階層完了

--- [16/30] 2026-03-10 ---
  ✅ 5階層完了

--- [17/30] 2026-03-11 ---
  ✅ 5階層完了

--- [18/30] 2026-03-12 ---
  ✅ 5階層完了

--- [19/30] 2026-03-13 ---
  ✅ 5階層完了

--- [20/30] 2026-03-14 ---
  ✅ 5階層完了

--- [21/30] 2026-03-15 ---
  ✅ 5階層完了

--- [22/30] 2026-03-16 ---
  ✅ 5階層完了

--- [23/30] 2026-03-17 ---
  ✅ 5階層完了

--- [24/30] 2026-03-18 ---
  ✅ 5階層完了

--- [25/30] 2026-03-19 ---
  ✅ 5階層完了

--- [26/3